In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# --- 1. Load 2023 Data ---
print("Loading 2023...")
c_23 = pd.read_csv("caract-2023.csv", sep=";", encoding="utf-8")
l_23 = pd.read_csv("lieux-2023.csv", sep=";", encoding="utf-8")
u_23 = pd.read_csv("usagers-2023.csv", sep=";", encoding="utf-8")
v_23 = pd.read_csv("vehicules-2023.csv", sep=";", encoding="utf-8")

# --- 2. Load 2024 Data ---
# (Adjust filenames if yours are different, e.g. 'Caract_2024.csv')
print("Loading 2024...")
c_24 = pd.read_csv("caract-2024.csv", sep=";", encoding="utf-8")
l_24 = pd.read_csv("lieux-2024.csv", sep=";", encoding="utf-8")
u_24 = pd.read_csv("usagers-2024.csv", sep=";", encoding="utf-8")
v_24 = pd.read_csv("vehicules-2024.csv", sep=";", encoding="utf-8")

# --- 3. Concatenate (Stacking them) ---
print("Concatenating years...")

# We use ignore_index=True to reset the index (0 to N) for the new combined tables
df_caract = pd.concat([c_23, c_24], ignore_index=True)
df_lieux = pd.concat([l_23, l_24], ignore_index=True)
df_usagers = pd.concat([u_23, u_24], ignore_index=True)
df_vehicules = pd.concat([v_23, v_24], ignore_index=True)

# Free up memory by deleting the yearly variables
del c_23, c_24, l_23, l_24, u_23, u_24, v_23, v_24

# --- 4. Verify ---
print(f"Combined Characteristics Shape: {df_caract.shape}")
print(f"Combined Usagers Shape: {df_usagers.shape}")

# Check if we have both years in the data
print("Years present:", df_caract['an'].unique())

Loading 2023...


C:\Users\guscr\AppData\Local\Temp\ipykernel_12396\2389747566.py:4: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  l_23 = pd.read_csv("lieux-2023.csv", sep=";", encoding="utf-8")


Loading 2024...
Concatenating years...
Combined Characteristics Shape: (109224, 15)
Combined Usagers Shape: (250976, 16)
Years present: [2023 2024]


In [4]:
# Check for duplicates
dupes = df_lieux['Num_Acc'].duplicated().sum()
print(f"Duplicates in Lieux: {dupes}")

# STRATEGY: Keep the first occurrence.
# We explicitly sort by Num_Acc just to be stable, then pick the first.
df_lieux = df_lieux.sort_values('Num_Acc').drop_duplicates(subset=['Num_Acc'], keep='first')

print(f"New Shape after deduplication: {df_lieux.shape}")

Duplicates in Lieux: 31884
New Shape after deduplication: (109224, 18)


# Merge (Characteristics) and Location (Lieux)

In [5]:
# These are 1-to-1 relationships, so we can merge them directly.
df_master = pd.merge(df_caract, df_lieux, on='Num_Acc', how='inner')

print(f"Base shape after merging Caract + Lieux: {df_master.shape}")

Base shape after merging Caract + Lieux: (109224, 32)


In [8]:
display(df_master.head())

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
0,202300000001,7,5,2023,06:00,1,75,75101,2,4,...,1,-1,-1,1,NaN,-1,2,0,1,30
1,202300000002,7,5,2023,05:30,5,94,94080,2,1,...,1,-1,-1,1,NaN,-1,2,0,1,50
2,202300000003,7,5,2023,20:50,1,94,94022,2,3,...,1,1,0,1,NaN,-1,2,5,1,50
3,202300000004,6,5,2023,23:57,5,94,94078,2,1,...,1,18,1,1,NaN,12,2,0,1,50
4,202300000005,7,5,2023,00:50,5,94,94068,2,2,...,1,-1,-1,1,NaN,-1,2,0,1,30


# Aggregating Data vehicles


In [6]:
veh_counts = df_vehicules.groupby('Num_Acc').size().reset_index(name='nb_vehicules')

In [7]:
display(veh_counts.head())

,Num_Acc,nb_vehicules
0,202300000001,1
1,202300000002,1
2,202300000003,2
3,202300000004,3
4,202300000005,2


In [10]:
# Merge count into the master
df_master = pd.merge(df_master, veh_counts, on='Num_Acc', how='left')

# Target Variable

In [11]:
# The goal is to predict if an accident is "Severe".
# grav codes: 1=Uninjured, 2=Killed, 3=Hospitalized, 4=Slightly Injured
# Logic: If ANYONE in the accident is Killed (2) or Hospitalized (3), the accident is Severe (1).

def check_severe(series):
    if 2 in series.values or 3 in series.values:
        return 1
    return 0

# Group by Accident ID and apply the logic
severity = df_usagers.groupby('Num_Acc')['grav'].apply(check_severe).reset_index(name='target_severe')
df_master = pd.merge(df_master, severity, on='Num_Acc', how='inner')

In [12]:
display(severity.head())

,Num_Acc,target_severe
0,202300000001,0
1,202300000002,1
2,202300000003,1
3,202300000004,0
4,202300000005,0


# Cleaning

In [13]:
# A. Fix Dates (Create a real datetime column)
# Format 'hrmn' (e.g., 830 -> 0830, 1 -> 0001)
df_master['hrmn'] = df_master['hrmn'].astype(str).str.replace(':', '')
df_master['hrmn'] = df_master['hrmn'].apply(lambda x: x.zfill(4))

# Create the date string
date_strings = (df_master['an'].astype(str) + '-' + 
                df_master['mois'].astype(str) + '-' + 
                df_master['jour'].astype(str) + ' ' + 
                df_master['hrmn'].str[:2] + ':' + df_master['hrmn'].str[2:])

# Convert to datetime
df_master['datetime'] = pd.to_datetime(date_strings, errors='coerce')

# Extract clean features
df_master['hour'] = df_master['datetime'].dt.hour
df_master['month'] = df_master['datetime'].dt.month
df_master['day_of_week'] = df_master['datetime'].dt.dayofweek # 0=Monday

# B. Fix Coordinates (Strings with commas -> Floats)
def clean_coord(x):
    if isinstance(x, str):
        return float(x.replace(',', '.'))
    return x

if 'lat' in df_master.columns and 'long' in df_master.columns:
    df_master['lat'] = df_master['lat'].apply(clean_coord)
    df_master['long'] = df_master['long'].apply(clean_coord)

# C. Handle Missing Values (-1)
# Drop rows where critical info like Weather (atm) or Lighting (lum) is unknown
cols_to_clean = ['atm', 'lum', 'agg', 'int']
for col in cols_to_clean:
    if col in df_master.columns:
        df_master = df_master[df_master[col] != -1]

In [14]:
display(df_master.head())

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,surf,infra,situ,vma,nb_vehicules,target_severe,datetime,hour,month,day_of_week
0,202300000001,7,5,2023,0600,1,75,75101,2,4,...,2,0,1,30,1,0,2023-05-07 06:00:00,6,5,6
1,202300000002,7,5,2023,0530,5,94,94080,2,1,...,2,0,1,50,1,1,2023-05-07 05:30:00,5,5,6
2,202300000003,7,5,2023,2050,1,94,94022,2,3,...,2,5,1,50,2,1,2023-05-07 20:50:00,20,5,6
3,202300000004,6,5,2023,2357,5,94,94078,2,1,...,2,0,1,50,3,0,2023-05-06 23:57:00,23,5,5
4,202300000005,7,5,2023,0050,5,94,94068,2,2,...,2,0,1,30,2,0,2023-05-07 00:50:00,0,5,6


In [55]:
# --- 4. Save to CSV (Checkpoint) ---
# We save this so you can open it in a clean notebook for EDA
output_filename = 'france_accidents_2023_2024_cleaned.csv'
df_master.to_csv(output_filename, index=False)

print(f"Cleaning complete. File saved as: {output_filename}")
print(f"Final Shape: {df_master.shape}")
print(df_master[['datetime', 'lat', 'long', 'atm', 'target_severe']].head())

Cleaning complete. File saved as: france_accidents_2023_2024_cleaned.csv
Final Shape: (109214, 38)
             datetime        lat      long  atm  target_severe
0 2023-05-07 06:00:00  48.866386  2.323471    2              0
1 2023-05-07 05:30:00  48.845478  2.428681    3              1
2 2023-05-07 20:50:00  48.762400  2.406550    2              1
3 2023-05-06 23:57:00  48.732484  2.446876    3              0
4 2023-05-07 00:50:00  48.785810  2.492170    3              0
